## 依赖导入

本例演示聚类与判别分析：系统聚类（`scipy.cluster.hierarchy`：`linkage`/`dendrogram`/`fcluster`，含马氏距离法）、K 均值聚类（`sklearn.cluster.KMeans`）、用轮廓系数与肘部法则选 K、判别分析（马氏距离判别、`sklearn` 的 `LinearDiscriminantAnalysis` 及交叉验证）。矩阵求逆用 `numpy.linalg.inv`；9.3 中读取用户输入的类数用 `ast.literal_eval` 安全解析（替代 `eval`）。

In [ ]:
import ast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as sch
from numpy.linalg import inv
from sklearn.cluster import KMeans
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score

## 全局绘图配置

设置全局 `plt.rc`：启用 TeX 渲染数学公式、字号 16，保证后续谱系图与曲线的公式标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=16)

## 9.1  系统聚类谱系图（原始数据）

将一维数据 $\{2,3,3.5,7,9\}$ 转为列向量，用 `sch.linkage` 做系统聚类，并以 $\omega_1..\omega_5$ 为标签绘制谱系图（dendrogram）（源程序 ex11_1_1.py）。

In [ ]:
a = np.array([[2, 3, 3.5, 7, 9]]).T
c = sch.linkage(a)
s = ["$\\omega_" + str(i + 1) + "$" for i in range(5)]
sch.dendrogram(c, labels=s)
plt.show()

## 9.1  系统聚类谱系图（距离向量输入）

改为直接由数据两两绝对差 `d` 构成 condensed 距离向量，再用 `sch.linkage(d)` 聚类，并以 $\omega_1..\omega_n$ 为标签绘制谱系图，作为与列向量输入方式的等价对照（源程序 ex11_1_2.py）。

In [ ]:
a = [2, 3, 3.5, 7, 9]
n = len(a)
d = [abs(a[i] - a[j]) for i in range(n - 1) for j in range(i + 1, n)]
c = sch.linkage(d)
s = ["$\\omega_" + str(i + 1) + "$" for i in range(n)]
sch.dendrogram(c, labels=s)
plt.show()

## 9.2  标准化后的系统聚类

读取 `data/11_2.txt`，对每一列做 0-1 标准化 `(a - a.min(axis=0)) / (a.max(axis=0) - a.min(axis=0))` 以消除量纲差异，再调用 `sch.linkage` 聚类，并以 $\omega_1..\omega_n$ 为标签绘制谱系图（源程序 ex11_2.py）。

In [ ]:
a = np.loadtxt("data/11_2.txt")
n = a.shape[0]
b = (a - a.min(axis=0)) / (a.max(axis=0) - a.min(axis=0))
z = sch.linkage(b)
s = ["$\\omega_" + str(i + 1) + "$" for i in range(n)]
sch.dendrogram(z, labels=s)
plt.show()

## 9.3  马氏距离系统聚类与类数选择

对 $\{2,3,3.5,7,9\}$ 用 `sch.linkage(a, 'complete', 'mahalanobis')`（最长距离法 + 马氏距离）聚类并绘制谱系图；随后由键盘输入类数 n0（用 `ast.literal_eval` 安全解析为数值，替代不安全的 `eval`），再用 `sch.fcluster(c, t=n0, criterion='maxclust')` 按最大类数切割，打印各样本的聚类结果（源程序 ex11_3.py）。

In [ ]:
a = np.array([[2, 3, 3.5, 7, 9]]).T
n = len(a)
c = sch.linkage(a, "complete", "mahalanobis")
s = ["$\\omega_" + str(i + 1) + "$" for i in range(n)]
sch.dendrogram(c, labels=s)
plt.show()
n0 = ast.literal_eval(input("请输入聚类的类数 n0:\n"))
cluster = sch.fcluster(c, t=n0, criterion="maxclust")
print("聚类的结果为：", cluster)

## 9.4  K 均值聚类（二维点集）

对二维点集 $\{(1,3),(1.5,3.2),(1.3,2.8),(3,1)\}$ 用 `KMeans(2).fit(a)` 构建并求解 2 类聚类模型，提取各样本的聚类标签 `labels_` 与聚类中心 `cluster_centers_`（每一行是一个聚类中心）并打印（源程序 ex11_4.py）。

In [ ]:
a = np.array([[1, 3], [1.5, 3.2], [1.3, 2.8], [3, 1]])
md = KMeans(2).fit(a)
labels = md.labels_
centers = md.cluster_centers_
print(labels, "\n-----------\n", centers)

## 9.5  K 均值聚类（一维数据）

将一维数据 $\{2,3,3.5,7,9\}$ 转为列向量后用 `KMeans(2).fit(a)` 构造并求解 2 类聚类，打印聚类标签与聚类中心（每行一个中心），与 9.1 的系统聚类结果相互对照（源程序 ex11_5.py）。

In [ ]:
a = np.array([[2, 3, 3.5, 7, 9]]).T
md = KMeans(2).fit(a)
labels = md.labels_
centers = md.cluster_centers_
print(labels, "\n-----------\n", centers)

## 9.6  肘部法则（SSE 随类数 K 变化）

读取并标准化 `data/11_2.txt`，对类数 $K=1..\ n$ 依次做 `KMeans(K).fit(b)`，记录每个模型的惯性（组内平方和 `inertia_`）到 SSE，再绘制 SSE-K 折线，用“肘部”位置判断合适的聚类数（源程序 ex11_6.py）。

In [ ]:
a = np.loadtxt("data/11_2.txt")
b = (a - a.min(axis=0)) / (a.max(axis=0) - a.min(axis=0))
SSE = []
K = range(1, len(a) + 1)
for i in K:
    md = KMeans(i).fit(b)
    SSE.append(md.inertia_)
fig, ax = plt.subplots()
ax.plot(K, SSE, "*-")
plt.show()

## 9.7  轮廓系数选择类数

标准化 `data/11_2.txt` 后，对 $K=2..n-1$ 依次做 KMeans 并取标签，计算每个 K 的轮廓系数 `silhouette_score(b, labels)`，绘制轮廓系数-K 折线，取轮廓系数最大者为最佳聚类数（源程序 ex11_7.py）。

In [ ]:
a = np.loadtxt("data/11_2.txt")
b = (a - a.min(axis=0)) / (a.max(axis=0) - a.min(axis=0))
S = []
K = range(2, len(a))
for i in K:
    md = KMeans(i).fit(b)
    labels = md.labels_
    S.append(silhouette_score(b, labels))
fig, ax = plt.subplots()
ax.plot(K, S, "*-")
plt.show()

## 9.8  按相关系数做变量聚类

读取 `data/11_8.xlsx` 的（相关系数）矩阵并转置，用 `np.triu(b, k=1)` 取对角线上方元素 $r$，以距离 $d=1-|r|$ 做最长距离法系统聚类 `sch.linkage(d,'complete')`，并以 `range(1,15)` 作为变量标签绘制谱系图，实现变量聚类（源程序 ex11_8.py）。

In [ ]:
a = pd.read_excel("data/11_8.xlsx", header=None)
b = a.values.T
b = np.triu(b, k=1)
r = b[np.nonzero(b)]
d = 1 - abs(r)
z = sch.linkage(d, "complete")
sch.dendrogram(z, labels=range(1, 15))
plt.show()

## 9.10  马氏距离判别分析

读取 `data/11_10.xlsx`：前段为已知样本观测 $x_0$ 及其真实类别 $y_0$，末两行为待判样本 $x$（均转为 float）。以前 10 个已知样本为第 1 类、其余为第 2 类，分别求均值 mu1/mu2 与协方差阵 s1/s2；对待判样本与已知样本各计算到两类的马氏距离 $d=\sqrt{(x-\mu)^T s^{-1}(x-\mu)}$，按 `argmin+1` 取最近类判类。打印待判样本分类与已知样本回判结果，并计算已知样本的误判率 rate（真实类别与回判类别之差的比例）（源程序 ex11_10.py）。

In [ ]:
a = pd.read_excel("data/11_10.xlsx", header=None)
b = a.values
x0 = b[:-2, :-1].astype(float)
y0 = b[:-2, -1].astype(float)
x = b[-2:, :-1].astype(float)
A1 = x0[:10, :]
A2 = x0[10:16, :]
mu1 = A1.mean(axis=0)
mu2 = A2.mean(axis=0)
s1 = np.cov(A1.T, ddof=1)
s2 = np.cov(A2.T, ddof=1)
D = []
for i in x:
    d1 = np.sqrt((i - mu1) @ inv(s1) @ (i - mu1))
    d2 = np.sqrt((i - mu2) @ inv(s2) @ (i - mu2))
    D.append([d1, d2])
ind = np.argmin(D, axis=1) + 1
check = []
for i in x0:
    d1 = np.sqrt((i - mu1) @ inv(s1) @ (i - mu1))
    d2 = np.sqrt((i - mu2) @ inv(s2) @ (i - mu2))
    check.append([d1, d2])
ind2 = np.argmin(check, axis=1) + 1
rate = (y0 - ind2).sum() / len(y0)
print("待判样本的分类：", ind)
print("已知样本的检验：", ind2)

## 9.11  两总体线性判别（Fisher/LDA）

读取 `data/11_11.txt`：前 5 行为 A 类、次 5 行为 B 类已知样本，其余为待判样本 x。计算合并协方差阵 $V$（`np.cov(ddof=1)`）及其逆阵 $V^{-1}$，直接构造判别函数 $w(x)=k^Tx+b$，其中系数向量 $k=V^{-1}(\mu_1-\mu_2)$、常数项 $b=-\tfrac12(\mu_1+\mu_2)^TV^{-1}(\mu_1-\mu_2)$；由判别值 val 的正负判类。再用 `sklearn` 的 `LDA` 库函数拟合并取出 coef_、intercept_，验证其与手算 k、b 成同一比例 c（即两法等价），打印两法判别结果、k/b、k2/b2、c 及已知样本误判率（源程序 ex11_11.py）。

In [ ]:
a = np.loadtxt("data/11_11.txt")
a1 = a[:5, :]
a2 = a[5:10, :]
x = a[10:, :]
V = np.cov(a[:10, :].T, ddof=1)
VI = inv(V)
mu1 = a1.mean(axis=0)
mu2 = a2.mean(axis=0)
k = VI @ (mu1 - mu2)
b = -(mu1 + mu2) @ VI @ (mu1 - mu2) / 2
val = x @ k + b
print("判别函数的值：", val)
d = {0: "B", 1: "A"}
print("直接计算结果：", [d[e > 0] for e in val])
y0 = np.hstack([np.ones(5), np.zeros(5)])
md = LDA().fit(a[:10, :], y0)
k2 = md.coef_
b2 = md.intercept_
c = b2 / b
check = k * c
val2 = md.predict(x)
print("库函数结果：  ", [d[e] for e in val2])
print("k=", k, ",b=", b)
print("k2=", k2, ",b2=", b2)
print("比例 c=", c)
print("已知样本误判率为：", 1 - md.score(a[:10, :], y0))

## 9.12  判别分析的交叉验证

读取 `data/11_10.xlsx` 已知样本的特征 $x_0$ 与类别 $y_0$，用 `sklearn` 的 `LinearDiscriminantAnalysis` 配合 `cross_val_score(md, x0, y0, cv=2)` 做 2 折交叉验证，输出各折的准确率以评估判别模型的泛化能力（源程序 ex11_12.py）。

In [ ]:
a = pd.read_excel("data/11_10.xlsx", header=None)
b = a.values
x0 = b[:-2, :-1].astype(float)
y0 = b[:-2, -1].astype(float)
md = LDA()
print(cross_val_score(md, x0, y0, cv=2))